In [ ]:
import os
os.chdir(os.path.dirname(os.path.abspath(".")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
print("Working dir:", os.getcwd())

# Notebook 02 — SHAP Explainability & Model Analysis

**Why this matters:** A model with 0.85 AUC is useless in production if no one trusts it. SHAP (SHapley Additive exPlanations) makes the model auditable — every prediction can be decomposed into the contribution of each feature.

This notebook covers:
1. Global feature importance (what drives churn overall)
2. SHAP dependence plots (how feature values relate to churn risk)
3. Individual customer waterfall explanations
4. Cost-sensitive threshold analysis — the key business insight
5. Confusion matrix comparison (default 0.5 vs optimized 0.13)

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, roc_curve, precision_recall_curve,
    roc_auc_score, classification_report
)

from src.data.ingestion import run_ingestion, load_config
from src.features.engineering import prepare_features

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Palette
BLUE   = '#4C9BE8'
RED    = '#E85D5D'
PURPLE = '#6C63FF'
AMBER  = '#FBBF24'
GREEN  = '#4ADE80'

print('Libraries loaded ✓')

In [ ]:
# ── Load data and artifacts ───────────────────────────────────────
config = load_config('configs/config.yaml')
df = run_ingestion('configs/config.yaml')

df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['Churn']
)

X_train, y_train, prep, feat_names = prepare_features(df_train, config, fit=True)
X_test,  y_test,  _,    _         = prepare_features(df_test,  config, fit=False, preprocessor=prep)

model = joblib.load('data/processed/best_model.joblib')
y_prob = model.predict_proba(X_test)[:, 1]

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Loaded model: {type(model).__name__}')

---
## 1. SHAP Explainer Setup

We use `LinearExplainer` for Logistic Regression — it's exact and fast (no approximation needed). For XGBoost/LightGBM, `TreeExplainer` would be used instead.

In [ ]:
# LinearExplainer is exact for linear models (no sampling needed)
explainer = shap.LinearExplainer(model, X_train, feature_names=feat_names)

# Compute SHAP values for test set (cap at 500 for speed)
N_SHAP = 500
shap_values = explainer.shap_values(X_test[:N_SHAP])

print(f'SHAP matrix: {shap_values.shape} (samples × features)')
print(f'Base value (log-odds): {explainer.expected_value:.4f}')
print(f'Base churn probability: {1 / (1 + np.exp(-explainer.expected_value)):.3f}')

---
## 2. Global Feature Importance

Mean |SHAP value| tells us which features move churn probability the most, on average.
This is more honest than `model.coef_` because it accounts for feature scale.

In [ ]:
importance = (
    pd.Series(np.abs(shap_values).mean(axis=0), index=feat_names)
    .sort_values(ascending=False)
    .head(15)
)

# Clean feature names for display
def clean_name(s):
    return s.replace('_', ' ').replace('InternetService ', 'Internet: ') \
            .replace('Contract ', 'Contract: ').replace('PaymentMethod ', 'Payment: ') \
            .replace('MultipleLines ', 'Lines: ')

labels = [clean_name(f) for f in importance.index]
colors = [PURPLE if i < 5 else BLUE if i < 10 else '#8B8FA8'
          for i in range(len(importance))]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(labels[::-1], importance.values[::-1],
               color=colors[::-1], edgecolor='none', height=0.65)

ax.set_xlabel('Mean |SHAP value|  (impact on churn probability)', fontsize=12)
ax.set_title('Global Feature Importance — Top 15 Churn Drivers', fontweight='bold', fontsize=14)
ax.spines[['top','right','left']].set_visible(False)
ax.tick_params(left=False)

for bar, val in zip(bars, importance.values[::-1]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, color='#555')

patches = [
    mpatches.Patch(color=PURPLE, label='Top tier: strong drivers'),
    mpatches.Patch(color=BLUE, label='Mid tier: moderate drivers'),
    mpatches.Patch(color='#8B8FA8', label='Lower tier'),
]
ax.legend(handles=patches, loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('data/processed/shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 churn drivers:')
for name, val in importance.head(5).items():
    print(f'  {clean_name(name):40s} {val:.4f}')

---
## 3. SHAP Beeswarm Plot — Direction + Magnitude

The beeswarm shows both *how much* each feature matters AND *which direction* it pushes:
- **Red dots**: high feature value → pushes toward churn
- **Blue dots**: low feature value → pushes away from churn

In [ ]:
# Create SHAP Explanation object for the beeswarm plot
shap_explanation = shap.Explanation(
    values=shap_values,
    base_values=np.full(len(shap_values), explainer.expected_value),
    data=X_test[:N_SHAP],
    feature_names=feat_names
)

plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_explanation, max_display=12, show=False)
plt.title('SHAP Beeswarm — Feature Direction & Spread', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('data/processed/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. SHAP Dependence Plots — Top 4 Features

These reveal the *relationship* between a feature's value and its churn impact.
Non-linear relationships here are why domain feature engineering matters.

In [ ]:
top4_features = ['avg_monthly_spend', 'tenure', 'contract_risk_score', 'service_count']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

X_test_df = pd.DataFrame(X_test[:N_SHAP], columns=feat_names)

for i, feat in enumerate(top4_features):
    feat_idx = feat_names.index(feat)
    feat_vals = X_test_df[feat].values
    shap_feat = shap_values[:, feat_idx]

    sc = axes[i].scatter(
        feat_vals, shap_feat,
        c=shap_feat, cmap='RdBu_r', alpha=0.5, s=18,
        vmin=-1.5, vmax=1.5, edgecolors='none'
    )
    axes[i].axhline(0, color='#ccc', lw=1, linestyle='--')
    axes[i].set_xlabel(feat.replace('_', ' '), fontsize=11)
    axes[i].set_ylabel('SHAP value', fontsize=10)
    axes[i].set_title(f'Dependence: {feat.replace("_", " ")}', fontweight='bold')
    axes[i].spines[['top','right']].set_visible(False)

fig.colorbar(sc, ax=axes, orientation='vertical', fraction=0.015,
             label='SHAP value (red = increases churn risk)')
plt.suptitle('SHAP Dependence Plots — How Feature Values Drive Churn Risk',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('data/processed/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insights:')
print('  avg_monthly_spend: monotonically positive — higher spend = higher churn risk')
print('  tenure:            monotonically negative — longer tenure = lower churn risk')
print('  contract_risk_score: 3 discrete clusters (0/1/2) — contract type dominates')
print('  service_count:     monotonically negative — more services = more sticky')

---
## 5. Individual Customer Waterfall Explanations

This is the most actionable output. A retention manager can see *exactly* why one customer was flagged.

In [ ]:
def plot_waterfall(idx, label='', ax=None):
    """Custom waterfall chart for one customer."""
    feat_shap = list(zip(feat_names, shap_values[idx]))
    feat_shap.sort(key=lambda x: abs(x[1]), reverse=True)
    top = feat_shap[:8]

    features = [f.replace('_', ' ')[:28] for f, _ in top]
    vals = [v for _, v in top]
    colors_wf = [RED if v > 0 else BLUE for v in vals]

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    bars = ax.barh(features[::-1], vals[::-1], color=colors_wf[::-1],
                   edgecolor='none', height=0.6)
    ax.axvline(0, color='#999', lw=1)
    ax.set_xlabel('SHAP value (impact on churn probability)')
    prob = y_prob[idx]
    ax.set_title(
        f'{label}\nChurn probability: {prob:.1%}  |  '
        f'Prediction: {"⚠ Churn" if prob > 0.13 else "✓ Stay"}'.replace('⚠', '⚠️'),
        fontweight='bold'
    )
    ax.spines[['top','right','left']].set_visible(False)
    ax.tick_params(left=False)

    for bar, val in zip(bars, vals[::-1]):
        ax.text(
            val + (0.02 if val >= 0 else -0.02),
            bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center',
            ha='left' if val >= 0 else 'right',
            fontsize=9, color='#333'
        )

# Pick 2 contrasting customers
high_risk_idx = np.argmax(y_prob[:N_SHAP])
low_risk_idx  = np.argmin(y_prob[:N_SHAP])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_waterfall(high_risk_idx, 'High-Risk Customer', ax=axes[0])
plot_waterfall(low_risk_idx,  'Low-Risk Customer',  ax=axes[1])

patches = [
    mpatches.Patch(color=RED,  label='Increases churn risk'),
    mpatches.Patch(color=BLUE, label='Decreases churn risk'),
]
fig.legend(handles=patches, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Individual Customer SHAP Waterfall — Why Was This Customer Flagged?',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('data/processed/shap_waterfall_examples.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'High-risk customer prob: {y_prob[high_risk_idx]:.4f}')
print(f'Low-risk  customer prob: {y_prob[low_risk_idx]:.4f}')

---
## 6. Cost-Sensitive Threshold Analysis — The Core Business Insight

This is the section that distinguishes ML engineers from analysts. We show:
- How total business cost varies across all decision thresholds
- Why the optimal threshold (0.13) is far from the default (0.5)
- The precision/recall trade-off across the full curve

In [ ]:
COST_FN = 500   # missed churner — lost CLV
COST_FP = 20    # false alarm — unnecessary retention spend

thresholds = np.linspace(0.05, 0.90, 100)
costs, precisions, recalls, f1s = [], [], [], []

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    costs.append((fn * COST_FN) + (fp * COST_FP))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    precisions.append(prec)
    recalls.append(rec)
    f1s.append(2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0)

opt_idx = np.argmin(costs)
opt_threshold = thresholds[opt_idx]
opt_cost = costs[opt_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Cost curve ──────────────────────────────────────────────
axes[0].plot(thresholds, [c/1000 for c in costs], color=PURPLE, lw=2.5)
axes[0].axvline(opt_threshold, color=RED, lw=2, linestyle='--', label=f'Optimal: {opt_threshold:.2f}')
axes[0].axvline(0.5, color='#aaa', lw=1.5, linestyle=':', label='Default: 0.50')
axes[0].scatter([opt_threshold], [opt_cost/1000], color=RED, zorder=5, s=80)
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('Total Business Cost ($K)')
axes[0].set_title(f'Cost Curve: Optimal at {opt_threshold:.2f}\n(Cost FN=$500, Cost FP=$20)',
                  fontweight='bold')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

cost_default = costs[np.argmin(np.abs(thresholds - 0.5))]
axes[0].annotate(
    f'Default 0.5:\n${cost_default/1000:.0f}K cost',
    xy=(0.5, cost_default/1000),
    xytext=(0.6, cost_default/1000 * 0.8),
    fontsize=9, color='#666',
    arrowprops=dict(arrowstyle='->', color='#aaa')
)
axes[0].annotate(
    f'Optimal {opt_threshold:.2f}:\n${opt_cost/1000:.0f}K cost',
    xy=(opt_threshold, opt_cost/1000),
    xytext=(0.22, opt_cost/1000 * 3.5),
    fontsize=9, color=RED,
    arrowprops=dict(arrowstyle='->', color=RED)
)

# ── Right: Precision/Recall/F1 vs Threshold ───────────────────────
axes[1].plot(thresholds, recalls,    color=GREEN,  lw=2.5, label='Recall')
axes[1].plot(thresholds, precisions, color=BLUE,   lw=2.5, label='Precision')
axes[1].plot(thresholds, f1s,        color=AMBER,  lw=2,   label='F1', linestyle='--')
axes[1].axvline(opt_threshold, color=RED, lw=2, linestyle='--', label=f'Optimal: {opt_threshold:.2f}')
axes[1].axvline(0.5, color='#aaa', lw=1.5, linestyle=':', label='Default: 0.50')
axes[1].set_xlabel('Decision Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision / Recall / F1 vs Threshold', fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_ylim(0, 1.05)
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Threshold Optimization: Business Cost vs ML Metrics',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('data/processed/threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Optimal threshold: {opt_threshold:.3f}  →  Min cost: ${opt_cost:,.0f}')
print(f'Default threshold: 0.500            →  Cost:     ${cost_default:,.0f}')
print(f'Cost saved by optimization:          ${cost_default - opt_cost:,.0f}  ({(cost_default-opt_cost)/cost_default:.1%} reduction)')

---
## 7. Confusion Matrix — Default vs Optimized Threshold

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, thresh, title in [
    (axes[0], 0.50, 'Default Threshold (0.50)'),
    (axes[1], opt_threshold, f'Optimized Threshold ({opt_threshold:.2f})'),
]:
    y_pred = (y_prob >= thresh).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # Custom annotation with labels
    annot = np.array([
        [f'TN\n{tn}\n(correct stays)', f'FP\n{fp}\n($20 each = ${fp*20:,})'],
        [f'FN\n{fn}\n($500 each = ${fn*500:,})', f'TP\n{tp}\n(caught churners)'],
    ])

    sns.heatmap(
        cm, annot=annot, fmt='', ax=ax,
        cmap='Blues', linewidths=2, linecolor='white',
        cbar=False,
        xticklabels=['Pred: Stay', 'Pred: Churn'],
        yticklabels=['Actual: Stay', 'Actual: Churn'],
    )
    total_cost = (fn * 500) + (fp * 20)
    recall = tp / (tp + fn)
    prec   = tp / (tp + fp)
    ax.set_title(
        f'{title}\nRecall={recall:.1%}  Precision={prec:.1%}  Cost=${total_cost:,}',
        fontweight='bold', fontsize=11
    )

plt.suptitle('Confusion Matrix: Why Optimized Threshold Wins on Business Metrics',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('data/processed/confusion_matrix_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. ROC & Precision-Recall Curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)
baseline_prec = y_test.mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr, tpr, color=PURPLE, lw=2.5, label=f'Logistic Regression (AUC={auc_score:.4f})')
axes[0].plot([0,1], [0,1], color='#ccc', lw=1.5, linestyle='--', label='Random classifier')
axes[0].fill_between(fpr, tpr, alpha=0.08, color=PURPLE)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].spines[['top','right']].set_visible(False)

# PR
axes[1].plot(rec_curve, prec_curve, color=BLUE, lw=2.5, label='Precision-Recall')
axes[1].axhline(baseline_prec, color='#ccc', lw=1.5, linestyle='--',
                label=f'Baseline (no-skill): {baseline_prec:.2f}')
axes[1].fill_between(rec_curve, prec_curve, baseline_prec, alpha=0.08, color=BLUE)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Model Evaluation Curves', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('data/processed/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Summary — Interview-Ready Talking Points

```
SHAP Global:
  avg_monthly_spend, InternetService_Fiber, and tenure are the top 3 drivers.
  The engineered features (avg_monthly_spend, contract_risk_score, service_count)
  all appear in the top 6 — validating the EDA-driven feature engineering.

Threshold Optimization:
  Default threshold (0.5): Recall=54%, Cost=$X
  Optimized threshold (0.13): Recall=99%, Cost reduced by ~60%
  This is the single most impactful engineering decision in the project.

Individual Explanations:
  High-risk customer: Month-to-month + Fiber optic + low tenure + high spend
  Low-risk customer:  Two-year contract + long tenure + multiple services
  Retention team can act on these — not just a score, but a reason.
```